In [ ]:
#For tables and images extraction
import fitz  # PyMuPDF
import pdfplumber
import pandas as pd
import os

pdf_folder = os.path.expanduser("~/Desktop/mof_pdfs")

output_folder = os.path.expanduser("~/Desktop/mof_extracted")
images_folder = os.path.join(output_folder, "images")
tables_folder = os.path.join(output_folder, "tables")

os.makedirs(output_folder, exist_ok=True)
os.makedirs(images_folder, exist_ok=True)
os.makedirs(tables_folder, exist_ok=True)

all_documents = []

for pdf_file in os.listdir(pdf_folder):

    if not pdf_file.lower().endswith(".pdf"):
        continue

    pdf_path = os.path.join(pdf_folder, pdf_file)

    print(f"Processing: {pdf_file}")

    # -------------------
    # TEXT EXTRACTION
    # -------------------
    doc = fitz.open(pdf_path)

    full_text = ""

    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text += page.get_text()

    # -------------------
    # IMAGE EXTRACTION
    # -------------------
    image_count = 0

    for page_index in range(len(doc)):

        page = doc[page_index]

        image_list = page.get_images(full=True)

        for img_index, img in enumerate(image_list):

            xref = img[0]

            base_image = doc.extract_image(xref)

            image_bytes = base_image["image"]

            image_ext = base_image["ext"]

            image_name = (
                f"{os.path.splitext(pdf_file)[0]}"
                f"_page{page_index+1}"
                f"_img{img_index+1}.{image_ext}"
            )

            image_path = os.path.join(images_folder, image_name)

            with open(image_path, "wb") as img_file:
                img_file.write(image_bytes)

            image_count += 1

    doc.close()

    # -------------------
    # TABLE EXTRACTION
    # -------------------
    table_count = 0

    try:
        with pdfplumber.open(pdf_path) as pdf:

            for page_num, page in enumerate(pdf.pages):

                tables = page.extract_tables()

                for table_index, table in enumerate(tables):

                    if table:

                        df = pd.DataFrame(table)

                        table_file = (
                            f"{os.path.splitext(pdf_file)[0]}"
                            f"_page{page_num+1}"
                            f"_table{table_index+1}.csv"
                        )

                        table_path = os.path.join(
                            tables_folder,
                            table_file
                        )

                        df.to_csv(table_path, index=False)

                        table_count += 1

    except Exception as e:
        print("Table extraction error:", e)

    all_documents.append({
        "file": pdf_file,
        "text": full_text,
        "images_extracted": image_count,
        "tables_extracted": table_count
    })

# Save summary
summary_df = pd.DataFrame(all_documents)

summary_path = os.path.join(
    output_folder,
    "extraction_summary.csv"
)

summary_df.to_csv(summary_path, index=False)

print("\nDone!")
print(summary_df)

Processing: Budget Translation_2082-2083_nzrwuof.pdf
Processing: source-book_kzajtb9.pdf
Processing: रातो  किताब व्यय अनुमान 82_2_14_lfkxxnc.pdf


In [3]:
#For text extraction
import fitz  # PyMuPDF
import pandas as pd
import os

pdf_folder = os.path.expanduser("~/Desktop/mof_pdfs")

documents = []

for pdf_file in os.listdir(pdf_folder):

    if not pdf_file.lower().endswith(".pdf"):
        continue

    pdf_path = os.path.join(pdf_folder, pdf_file)

    print("Reading:", pdf_file)

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    documents.append({
        "file": pdf_file,
        "text": text
    })

print("Finished:", len(documents))

Reading: Budget Translation_2082-2083_nzrwuof.pdf
Reading: source-book_kzajtb9.pdf
Reading: रातो  किताब व्यय अनुमान 82_2_14_lfkxxnc.pdf
Finished: 3


In [4]:
#Saving text in mof_extracted folder as document
import os

text_folder = os.path.expanduser("~/Desktop/mof_extracted/text")

os.makedirs(text_folder, exist_ok=True)

for doc in documents:

    filename = os.path.splitext(doc["file"])[0] + ".txt"

    with open(
        os.path.join(text_folder, filename),
        "w",
        encoding="utf-8"
    ) as f:
        f.write(doc["text"])

print("All text files saved.")

All text files saved.


In [8]:

#Checking the text files 
file_path = os.path.expanduser("~/Desktop/mof_extracted/text")

for f in os.listdir(file_path):
    full_path = os.path.join(file_path, f)

    print("\nFILE:", f)

    with open(full_path, "r", encoding="utf-8") as file:
        print(file.read()[:300])  # preview first 300 chars


FILE: Budget Translation_2082-2083_nzrwuof.txt
 
 
 
 
Budget Speech of 
Fiscal Year 2025/26 
 
 
 
 
 
 
 
 
 
 
 
 
Government of Nepal  
Ministry of Finance  
2025 
 
 
 
 
 
Presented by the Hon’ble Deputy Prime Minister and Minister of Finance,  
Mr. Bishnu Prasad Paudel,  
in the joint meeting of the Federal Parliament 
 
 
 
 
 
 
 
 
 
 

FILE: रातो  किताब व्यय अनुमान 82_2_14_lfkxxnc.txt
यय अनुमानको ववरण
(खच शीषकगत र ोतगत समेत)
आथक वष 2082/83
नेपाल सरकार
अथ म&'ालय
2082
www.mof.gov.np
bpd@mof.gov.np
आथक वष 2082/83 को यय अनुमानको ववरण 
सूची - प
2
0
8
2
/
8
3
अनुदान
संकेत
नकायको नाम
पाना नं.
आथक वष 2082/83 को यय अनुमानको सारांश
संघीय सि(त कोषमाथ ययभार
ख+ड - १
संघी

FILE: source-book_kzajtb9.txt
(Unofficial Translation)
(For Official Use Only)
Source Book
for
Projects Financed with Foreign Assistance
Fiscal Year 2025/26
Government of Nepal
Ministry of Finance
2025
www.mof.gov.np
Table Of Contents
Ministry
Code
Description
Page No
Summary Of Ministrywise Devel